# Week 7: the full pipeline from YAML

Normally you run these commands in a terminal (or in tmux on Hypatia). Here they run through `subprocess`
so you can see the outputs. The chain is the one you ran on TrackML: train a hit filter, write its
predictions for every split, train the tracker on the filtered hits, evaluate.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

In [ ]:
import subprocess

data_dir = ensure_toy_data(num_train=100 if FAST else 2000, num_val=20 if FAST else 200, num_test=20 if FAST else 200)
epochs = "1" if FAST else "10"


def run(*args):
    cmd = [sys.executable, "-m", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode:
        print(result.stderr[-3000:])
        raise RuntimeError("command failed")


run("minihep.run_tracking", "fit", "--config", "configs/toy_tracking.yaml", "--print_config")

In [ ]:
run("minihep.run_filter", "fit", "--config", "configs/toy_filter.yaml", "--trainer.max_epochs", epochs, "--data.num_val", "-1")
filter_run = max(Path("logs").glob("toy-filter_*"))
for split in ("train", "val", "test"):
    run("minihep.run_filter", "test", "--config", str(filter_run / "config.yaml"), "--data.test_dir", f"data/toy/{split}")
print(sorted(p.name for p in (filter_run / "ckpts").iterdir()))

In [ ]:
import pandas as pd

metrics = pd.read_csv(filter_run / "metrics.csv")
val = metrics.dropna(subset=["val/loss"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(val.epoch, val["val/loss"], "o-")
axes[0].set_title("filter val loss")
axes[1].plot(val.epoch, val["val/final_hit_filter_recall"], "o-", label="recall")
axes[1].plot(val.epoch, val["val/final_hit_filter_precision"], "o-", label="precision")
axes[1].legend()
plt.show()

In [ ]:
import h5py

h5 = {s: next((filter_run / "ckpts").glob(f"*__{s}.h5")) for s in ("train", "val", "test")}
with h5py.File(h5["test"]) as f:
    first = sorted(f)[0]
    f[first].visititems(lambda path, obj: print(f"{first}/{path}", getattr(obj, "shape", "")))
hit_eval = [a for s in ("train", "val", "test") for a in (f"--data.hit_eval_{s}", str(h5[s]))]
run(
    "minihep.run_tracking",
    "fit",
    "--config",
    "configs/toy_tracking.yaml",
    "--trainer.max_epochs",
    epochs,
    "--data.num_val",
    "-1",
    *hit_eval,
    "--data.hit_filter_threshold",
    "0.1",
)
track_run = max(Path("logs").glob("toy-tracking_*"))
run("minihep.run_tracking", "test", "--config", str(track_run / "config.yaml"))

In [ ]:
metrics = pd.read_csv(track_run / "metrics.csv")
val = metrics.dropna(subset=["val/loss"])
fig, axes = plt.subplots(1, 3, figsize=(17, 3.5))
axes[0].plot(val.epoch, val["val/loss"], "o-")
axes[0].set_title("val/loss")
for wp in (0.5, 0.75, 1.0):
    axes[1].plot(val.epoch, val[f"val/p{wp}_eff"], "o-", label=f"eff @ {wp}")
    axes[2].plot(val.epoch, val[f"val/p{wp}_pur"], "o-", label=f"pur @ {wp}")
axes[1].legend()
axes[2].legend()
plt.show()
print((track_run / "metadata.yaml").read_text())

## Your turn

1. Add `--config configs/comet.yaml` to the tracking fit (after putting your API key in `~/.comet.config`),
   and find the run in the Comet UI. Then rerun with `online: false` and upload the offline archive.
2. Override something deep from the command line, e.g. `--model.model.init_args.decoder.init_args.num_layers 4`
   (check the exact key with `--print_config`).
3. Write a small script that reads the tracking `__test.h5` and computes the double-majority efficiency
   per event from `preds/final/...` and `targets/...` using `minihep.metrics.double_majority`.